In [1]:
import pandas as pd
import snowflake.connector

In [2]:
def get_dt_formula(dt_flag, row):
    dt_formulas = {
        'ACCP6' : f'="TRY_TO_DATE("&J{row}&",\'0YYYYMM\')"',
        'DATS8' : f'="TRY_TO_DATE("&J{row}&",\'YYYYMMDD\')"',
        'DEC15' : f'="TRY_TO_TIMESTAMP_NTZ("&J{row}&", \'YYYYMMDDHH24MISS\')"',
        'DEC16' : f'="DATEADD(YEAR, 20, TO_TIMESTAMP_NTZ("&J{row}&"::NUMBER, 4))"',
        'DEC21' : f'="TRY_TO_TIMESTAMP_NTZ(REGEXP_SUBSTR("&J{row}&",\'/Date\\((\\d+)\\+\\d+\\)/\', 1, 1, \'e\'))"',
        'TIMS6' : f'="TRY_TO_TIME("&J{row}&",\'HH24MISS\')"'
    }
    return dt_formulas[dt_flag]

In [3]:
conn = snowflake.connector.connect(
    user = 'jay.spiers@us.tel.com',
    authenticator = 'externalbrowser',
    account = 'telus-global.privatelink',
    role = 'SYSADMIN_DEV',
    warehouse = 'DEVELOPER_WH',
    database = 'ODS',
    schema = 'INFORMATION_SCHEMA'
)

curs = conn.cursor()
tbl_names = curs.execute("select TABLE_NAME from TABLES where TABLE_SCHEMA = 'SAPAPI' and TABLE_NAME <> 'LIST_MISSING_KEYS'").fetch_pandas_all()
cds_views = {'_'.join(tbl.split('_')[3:]) : '_'.join(tbl.split('_')[:3]) for tbl in tbl_names['TABLE_NAME']}

In [4]:
path = 'C:\\Users\\223749\\Downloads\\'

typeDict = {
    'ACCP': 'VARCHAR(16777216)', # represents a month, needs to be parsed from string 0YYYYMM
    'CHAR': 'VARCHAR(16777216)',
    'CLNT': 'VARCHAR(16777216)',
    'CUKY': 'VARCHAR(16777216)',
    'CURR': 'NUMBER(38,10)',
    'DATS': 'VARCHAR(16777216)', # represents a date, needs to be parsed from string YYYYMMDD
    'DEC': 'NUMBER(38,10)', # sizes 15, 16, 21 are date/time formats
    'DF16_DEC': 'NUMBER(38,10)',
    'DF16_RAW': 'NUMBER(38,10)',
    'DF34_DEC': 'NUMBER(38,10)',
    'DF34_RAW': 'NUMBER(38,10)',
    'FLTP': 'NUMBER(38,10)',
    'INT1': 'NUMBER(38,0)',
    'INT2': 'NUMBER(38,0)',
    'INT4': 'NUMBER(38,0)',
    'INT8': 'NUMBER(38,0)',
    'LANG': 'VARCHAR(16777216)',
    'LCHR': 'VARCHAR(16777216)',
    'LRAW': 'VARCHAR(16777216)',
    'NUMC': 'VARCHAR(16777216)',
    'QUAN': 'NUMBER(38,10)',
    'RAW': 'VARCHAR(16777216)',
    'RAWSTRING': 'VARCHAR(16777216)',
    'SSTRING': 'VARCHAR(16777216)',
    'STRING': 'VARCHAR(16777216)',
    'TIMS': 'VARCHAR(16777216)', # represents a time, needs to be parsed from string HHMMSS
    'UNIT': 'VARCHAR(16777216)',
    'SSTRING': 'VARCHAR(16777216)',
    'LSTRING': 'VARCHAR(16777216)',
    'STEXT': 'VARCHAR(16777216)',
    'LTEXT': 'VARCHAR(16777216)',
    'KTEXT': 'VARCHAR(16777216)'
}



In [6]:
# download field info from within the table DD03VT from se16n

df = pd.read_excel(path+'fields.xlsx')

df = df.rename(columns={'No. of Characters':'Size', 'Key field': 'PK', 'Table position':'idx'})
df["CDS View"] = [cds_views[tbl] if tbl in cds_views else '' for tbl in df['Table Name']]

df["Mod Name"] = df["Short Description"].str.strip().str.upper()
df["Mod Name"] = df["Mod Name"].str.replace(r'\([^)]*\)', '', regex=True)
df["Mod Name"] = df["Mod Name"].str.replace(r'\W+', '_', regex=True)
df["Mod Name"] = df["Mod Name"].str.replace(r'^_|_$', '', regex=True)
df["Mod Name"] = df["Mod Name"].str.replace(r'^1ST', 'FIRST', regex=True)
df["Mod Name"] = df["Mod Name"].str.replace(r'^2ND', 'SECOND', regex=True)
df["Mod Name"] = df["Mod Name"].str.replace(r'^3RD', 'THIRD', regex=True)
df["Mod Name"] = df["Mod Name"].str.replace(r'^4TH', 'FOURTH', regex=True)

df["SF Type"] = [typeDict[d] if d in typeDict else 'VARCHAR(16777216)' for d in df['Data Type']]
df["isDuplicate"] = df.duplicated(subset=['Table Name', 'Mod Name'],keep=False)


for idx, row in df.iterrows():
    if row["isDuplicate"] is True:
        row["Mod Name"] = row["Mod Name"]+ "_" +row["Field Name"]
        df.at[idx,"Mod Name"] = row["Mod Name"]

    df.at[idx, 'source cte'] = f'="        "&LOWER(A{idx+2})&"."&C{idx+2}&" AS "&J{idx+2}&","'
    df.at[idx, 'transform cte'] = f'="        "&J{idx+2}&","'
    df.at[idx, 'type cte'] = f'="        CAST("&J{idx+2}&" AS "&K{idx+2}&") AS "&J{idx+2}&","'
    df.at[idx, 'snow'] = f'="    "&J{idx+2}&" "&K{idx+2}&","'

    df.at[idx, 'dt flag'] = ''
    df.at[idx, 'dt formula'] = ''
    if (row['Data Type']=='DEC' and row['Size']>=15 and row['Size']<=21) or (row['Data Type'] in ['ACCP', 'DATS', 'TIMS']):
        dt_flag = str(row['Data Type'])+str(row['Size'])
        df.at[idx, 'dt flag'] = dt_flag
        df.at[idx, 'dt formula'] = get_dt_formula(dt_flag, idx+2)
        df.at[idx, 'SF Type'] = 'VARCHAR(16777216)'

df = df[['Table Name', 'CDS View', 'Field Name', 'Short Description', 'Data Type', 'Size', 'PK', 'idx', 'Mod Name', 'SF Type', 'Mod Name', 'SF Type', 'isDuplicate', 'source cte', 'transform cte', 'type cte', 'snow', 'dt flag', 'dt formula']]
df = df.sort_values(by=['CDS View', 'idx']).reset_index(drop=True)
df.to_excel(path+'Common Field Names.xlsx', index=False)

In [7]:
df

,Table Name,CDS View,Field Name,Short Description,Data Type,Size,PK,idx,Mod Name,SF Type,Mod Name,SF Type,isDuplicate,source cte,transform cte,type cte,snow,dt flag,dt formula
0,VBAK,ZCDV_T_CMTU009,MANDT,Client,CLNT,3,X,1,CLIENT,VARCHAR(16777216),CLIENT,VARCHAR(16777216),False,"="" ""&LOWER(A8411)&"".""&C8411&"" AS ""&J841...","="" ""&J8411&"",""","="" CAST(""&J8411&"" AS ""&K8411&"") AS ""&J8...","="" ""&J8411&"" ""&K8411&"",""",,
1,VBAK,ZCDV_T_CMTU009,VBELN,Sales Document,CHAR,10,X,2,SALES_DOCUMENT,VARCHAR(16777216),SALES_DOCUMENT,VARCHAR(16777216),False,"="" ""&LOWER(A8474)&"".""&C8474&"" AS ""&J847...","="" ""&J8474&"",""","="" CAST(""&J8474&"" AS ""&K8474&"") AS ""&J8...","="" ""&J8474&"" ""&K8474&"",""",,
2,VBAK,ZCDV_T_CMTU009,ERDAT,Date on which the record was created,DATS,8,NaN,3,DATE_ON_WHICH_THE_RECORD_WAS_CREATED,VARCHAR(16777216),DATE_ON_WHICH_THE_RECORD_WAS_CREATED,VARCHAR(16777216),False,"="" ""&LOWER(A8348)&"".""&C8348&"" AS ""&J834...","="" ""&J8348&"",""","="" CAST(""&J8348&"" AS ""&K8348&"") AS ""&J8...","="" ""&J8348&"" ""&K8348&"",""",DATS8,"=""TRY_TO_DATE(""&J8348&"",'YYYYMMDD')"""
3,VBAK,ZCDV_T_CMTU009,ERZET,Entry time,TIMS,6,NaN,4,ENTRY_TIME,VARCHAR(16777216),ENTRY_TIME,VARCHAR(16777216),False,"="" ""&LOWER(A8350)&"".""&C8350&"" AS ""&J835...","="" ""&J8350&"",""","="" CAST(""&J8350&"" AS ""&K8350&"") AS ""&J8...","="" ""&J8350&"" ""&K8350&"",""",TIMS6,"=""TRY_TO_TIME(""&J8350&"",'HH24MISS')"""
4,VBAK,ZCDV_T_CMTU009,ERNAM,Name of Person who Created the Object,CHAR,12,NaN,5,NAME_OF_PERSON_WHO_CREATED_THE_OBJECT,VARCHAR(16777216),NAME_OF_PERSON_WHO_CREATED_THE_OBJECT,VARCHAR(16777216),False,"="" ""&LOWER(A8349)&"".""&C8349&"" AS ""&J834...","="" ""&J8349&"",""","="" CAST(""&J8349&"" AS ""&K8349&"") AS ""&J8...","="" ""&J8349&"" ""&K8349&"",""",,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10225,BUT020,ZCDV_T_CMTU209,DATE_FROM,Date Obsolete,DATS,8,NaN,12,DATE_OBSOLETE,VARCHAR(16777216),DATE_OBSOLETE,VARCHAR(16777216),False,"="" ""&LOWER(A1948)&"".""&C1948&"" AS ""&J194...","="" ""&J1948&"",""","="" CAST(""&J1948&"" AS ""&K1948&"") AS ""&J1...","="" ""&J1948&"" ""&K1948&"",""",DATS8,"=""TRY_TO_DATE(""&J1948&"",'YYYYMMDD')"""
10226,BUT020,ZCDV_T_CMTU209,ADDRESS_GUID,GUID of a Business Partner Address,RAW,16,NaN,13,GUID_OF_A_BUSINESS_PARTNER_ADDRESS,VARCHAR(16777216),GUID_OF_A_BUSINESS_PARTNER_ADDRESS,VARCHAR(16777216),False,"="" ""&LOWER(A1941)&"".""&C1941&"" AS ""&J194...","="" ""&J1941&"",""","="" CAST(""&J1941&"" AS ""&K1941&"") AS ""&J1...","="" ""&J1941&"" ""&K1941&"",""",,
10227,BUT020,ZCDV_T_CMTU209,ADDR_VALID_FROM,Validity Start of a Business Partner Address,DEC,15,NaN,14,VALIDITY_START_OF_A_BUSINESS_PARTNER_ADDRESS,VARCHAR(16777216),VALIDITY_START_OF_A_BUSINESS_PARTNER_ADDRESS,VARCHAR(16777216),False,"="" ""&LOWER(A1944)&"".""&C1944&"" AS ""&J194...","="" ""&J1944&"",""","="" CAST(""&J1944&"" AS ""&K1944&"") AS ""&J1...","="" ""&J1944&"" ""&K1944&"",""",DEC15,"=""TRY_TO_TIMESTAMP_NTZ(""&J1944&"", 'YYYYMMDDHH2..."
10228,BUT020,ZCDV_T_CMTU209,ADDR_VALID_TO,Validity End of a Business Partner Address,DEC,15,NaN,15,VALIDITY_END_OF_A_BUSINESS_PARTNER_ADDRESS,VARCHAR(16777216),VALIDITY_END_OF_A_BUSINESS_PARTNER_ADDRESS,VARCHAR(16777216),False,"="" ""&LOWER(A1945)&"".""&C1945&"" AS ""&J194...","="" ""&J1945&"",""","="" CAST(""&J1945&"" AS ""&K1945&"") AS ""&J1...","="" ""&J1945&"" ""&K1945&"",""",DEC15,"=""TRY_TO_TIMESTAMP_NTZ(""&J1945&"", 'YYYYMMDDHH2..."
